<a href="https://colab.research.google.com/github/karthikoo7/Machine_Learning-BDA-/blob/main/Regression%26Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('medical expenses.csv')
df.shape

(1338, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   expenses  1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [7]:
df.drop_duplicates(inplace=True)
df.shape

(1337, 7)

In [8]:
df.isna().sum().sum()

np.int64(0)

In [12]:
df.nunique()

,0
age,47
sex,2
bmi,275
children,6
smoker,2
region,4
expenses,1337


In [9]:
cont_col = ['age','bmi']

In [13]:
df_ohe = pd.get_dummies(df)
df_ohe.shape

(1337, 12)

Separate X & Y
--
target col --> expenses

In [14]:
X = df_ohe.drop(columns=['expenses'])
Y = df_ohe['expenses']

X.shape,Y.shape

((1337, 11), (1337,))

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
 Y.skew()

np.float64(1.5153909165486397)

In [17]:
X.skew()

,0
age,0.054781
bmi,0.284463
children,0.937421
sex_female,0.019469
sex_male,-0.019469
smoker_no,-1.463601
smoker_yes,1.463601
region_northeast,1.204009
region_northwest,1.204009
region_southeast,1.024467


In [18]:
# we dont stratify beacuse this is a regression problem

In [19]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=7)

X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((935, 11), (402, 11), (935,), (402,))

# Standardize the continous columns
--
only standardizes the data
--
does not make the data normally distributed --> POwer Transfrom used to make the data normally distributed

In [21]:
from sklearn.preprocessing import StandardScaler

In [22]:
sc = StandardScaler()
X_train[cont_col] = sc.fit_transform(X_train[cont_col])
X_test[cont_col] = sc.fit_transform(X_test[cont_col])

In [23]:
X_train[cont_col].mean()

,0
age,8.359326e-17
bmi,2.659786e-16


Linear Regression (Simple & Multiple)

In [20]:
from sklearn.linear_model import LinearRegression

# Evaluation metrics for Regresssion meodels

# Multiple linear regression done here -- multiple independent columns passed

In [25]:
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,r2_score

In [26]:
'''Ordinary least squares Linear Regression.
LinearRegression fits a linear model with coefficients w = (w1, ..., wp)
to minimize the residual sum of squares between the observed targets in
the dataset, and the targets predicted by the linear approximation.'''

lr = LinearRegression()
lr.fit(X_train,Y_train)
y_pred = lr.predict(X_test)


Y = mx + c --> b0 + bn * xn

In [28]:
lr.coef_ # (b1*x1+b2*x2+b3*x3....+bn*xn)

array([  3691.76308759,   2141.74157658,    550.95853205,     77.40983229,
          -77.40983229, -11932.614766  ,  11932.614766  ,    807.07943605,
           24.93368793,   -638.37864692,   -193.63447706])

In [29]:
lr.intercept_ # bo

np.float64(19626.04536523765)

In [27]:
print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))



root_mean_squared_error:  6205.744951484758
mean_absolute_error:  4324.344821439815
r2_score:  0.732803283122428


# Polynomial Regression --

1.add polynomial features to the data
--
Adds polynomial features in the data based on the degree
--
2.apply linear regression


In [44]:
from sklearn.preprocessing import PolynomialFeatures

In [45]:
pf = PolynomialFeatures(degree=2)
X_train_pf = pf.fit_transform(X_train)
X_test_pf = pf.fit_transform(X_test)

X_train_pf.shape,X_test_pf.shape

((935, 78), (402, 78))

In [46]:
pf3 = PolynomialFeatures(degree=3)
X_train_pf3 = pf3.fit_transform(X_train)
X_test_pf3 = pf3.fit_transform(X_test)

X_train_pf3.shape,X_test_pf3.shape

((935, 364), (402, 364))

In [47]:
lr = LinearRegression()
lr.fit(X_train_pf,Y_train)
y_pred = lr.predict(X_test_pf)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5159.1590662374365
mean_absolute_error:  3078.4237791139867
r2_score:  0.8153279762073066


In [48]:
lr = LinearRegression()
lr.fit(X_train_pf3,Y_train)
y_pred = lr.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5271.429826876078
mean_absolute_error:  3250.7226125012326
r2_score:  0.807203062336066


Regularization -- (Lasso - L1 Penalty)

In [49]:
from sklearn.linear_model import Lasso,Ridge

In [50]:
# for degree 2
ls = Lasso(alpha=1,random_state=7)
ls.fit(X_train_pf,Y_train)
y_pred = ls.predict(X_test_pf)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5157.897841220423
mean_absolute_error:  3078.176964313412
r2_score:  0.815418256233188


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.626e+08, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


In [51]:
# for degree 3
ls = Lasso(alpha=1,random_state=7)
ls.fit(X_train_pf3,Y_train)
y_pred = ls.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5261.760808943272
mean_absolute_error:  3239.9630598404397
r2_score:  0.8079096817778009


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.758e+08, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


Regularization -- (Ridge -- L2 Penalty)

In [52]:
# for degree 2
rd = Ridge(alpha=1,random_state=7)
rd.fit(X_train_pf,Y_train)
y_pred = rd.predict(X_test_pf)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5155.988370390011
mean_absolute_error:  3079.6969761754963
r2_score:  0.8155548964795364


In [53]:
# for degree 3
rd = Ridge(alpha=1,random_state=7)
rd.fit(X_train_pf3,Y_train)
y_pred = rd.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5262.165653417676
mean_absolute_error:  3244.4479287550294
r2_score:  0.8078801214467779


# Fine Tune alpha

# Lasso
1. use high alpha - 100
2. use low alpha -- 0.5

In [54]:
# for degree 3 ,, High alpha --> more features removed
ls = Lasso(alpha=100,random_state=7)
ls.fit(X_train_pf3,Y_train)
y_pred = ls.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5057.771077517304
mean_absolute_error:  3186.3182522366014
r2_score:  0.822515018559174


In [56]:
(ls.coef_ !=0).sum(),(ls.coef_ ==0).sum()  # columns selected,columns removed(i.e coeff set to 0)


(np.int64(33), np.int64(331))

In [57]:
# for degree 3 , low alpha --> less features removed
ls = Lasso(alpha=0.5,random_state=7)
ls.fit(X_train_pf3,Y_train)
y_pred = ls.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5266.88463230718
mean_absolute_error:  3246.020007892833
r2_score:  0.8075353903138774


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.475e+09, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


In [58]:
(ls.coef_ !=0).sum(),(ls.coef_ ==0).sum()  # columns selected, columns removed (i.e coeff set to 0


(np.int64(147), np.int64(217))

#Fine tuning on ridge

high aplha = 5
low alpha =0.5

In [66]:
# for degree 3
rd = Ridge(alpha=50,random_state=7)
rd.fit(X_train_pf3,Y_train)
y_pred = rd.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5135.016431008021
mean_absolute_error:  3245.5966519276844
r2_score:  0.8170523027479071


In [67]:
rd.coef_

array([    0.        ,   908.47886347,  1170.05457694,   132.69107054,
         -32.09420976,    32.09420976, -2112.90608063,  2112.90608063,
         187.20783025,   -90.25040742,   -42.6019531 ,   -54.35546973,
         186.2693733 ,    -9.45591171,   -82.97907211,   358.34489065,
         550.13397281,   494.26059506,   414.21826841,    54.27481875,
         229.01801153,   373.33641372,   251.84961946,   -71.83973524,
         103.76685007,   620.21482626,   549.83975068,  -943.12875596,
        2113.1833329 ,   260.09449411,   482.65155134,    92.39674714,
         334.91178435,   -23.23573252,   106.48730288,    26.20376766,
          79.62814729,    53.06292325,   233.49255411,   184.09961664,
        -150.49729529,  -134.40380492,   -32.09420976,     0.        ,
       -1016.52714647,   984.43293671,    67.07056576,    14.204182  ,
         -47.84377602,   -65.52518151,    32.09420976, -1096.37893416,
        1128.47314392,   120.13726449,  -104.45458942,     5.24182291,
      

In [68]:
(rd.coef_ == 0).sum()  # only making values smaller -- but not making them 0

np.int64(89)

In [61]:
# for degree 3
rd = Ridge(alpha=0.5,random_state=7)
rd.fit(X_train_pf3,Y_train)
y_pred = rd.predict(X_test_pf3)

print('root_mean_squared_error: ',root_mean_squared_error(Y_test,y_pred))
print('mean_absolute_error: ',mean_absolute_error(Y_test,y_pred))
print('r2_score: ',r2_score(Y_test,y_pred))


root_mean_squared_error:  5266.720729860989
mean_absolute_error:  3247.5045107501637
r2_score:  0.807547368905283


In [62]:
rd.coef_

array([ 0.00000000e+00,  1.07999938e+03,  1.49355547e+03,  1.72424943e+01,
       -6.40047928e+00,  6.40047928e+00, -2.49798050e+03,  2.49798050e+03,
        2.41383122e+02, -1.08399609e+02, -6.75447254e+01, -6.54387871e+01,
        4.39817304e+01,  3.38498382e+01, -2.27208096e+02,  4.06075443e+02,
        6.73923932e+02,  5.53958951e+02,  5.26040424e+02, -2.35506552e+00,
        2.75347240e+02,  5.34573167e+02,  2.72434033e+02, -1.89340623e+02,
       -1.22377354e+01,  7.66503986e+02,  7.27051485e+02, -1.22737102e+03,
        2.72092649e+03,  1.89721983e+02,  7.73706745e+02, -2.05552509e+01,
        5.50681993e+02,  7.13361088e+00,  9.57283749e+01, -7.84858807e+01,
        4.03681092e+02, -3.86438598e+02,  2.15525904e+02,  3.99095939e+02,
       -3.45543447e+02, -2.51835902e+02, -6.40047928e+00,  0.00000000e+00,
       -1.22943017e+03,  1.22302969e+03,  1.05461283e+02,  6.60946392e+01,
       -1.23608589e+02, -5.43478120e+01,  6.40047928e+00, -1.26855033e+03,
        1.27495081e+03,  

In [65]:
(rd.coef_ == 0).sum()

np.int64(89)